# nb_01_generate_source
**Simulated source system.** Writes the daily extract files for every business day that has
not been generated yet, up to `run_date`, into `lh_retail/Files/landing/<entity>/<date>/`.

* First run: generates `history_days` days of history (initial load on the first day).
* Later runs: generates only the missing days (catch-up). Re-running the same date does nothing.
* To start over: run `nb_00_setup` with `reset = True` (dev only).

In a real project this notebook would not exist: the files would come from the retailer's systems.

In [ ]:
run_date = ""            # business date to generate up to; "" = today (UTC)
history_days = 30        # days of history on the very first run
orders_per_day = 800
initial_customers = 2000
initial_products = 200
dq_error_rate = 0.01     # share of rows with deliberate data problems (0 = clean)
seed = 42
parent_run_id = ""

In [ ]:
%run nb_config

In [ ]:
%run nb_lib_logging

In [ ]:
%run nb_lib_generator

In [ ]:
LOCAL_ROOT = "/lakehouse/default"   # default lakehouse mounted as a local folder
STATE_FILE = f"{STATE_ROOT}/generator_state.json"

with RunLogger("nb_01_generate_source", run_date, parent_run_id) as run:
    if not os.path.isdir(os.path.join(LOCAL_ROOT, "Files")):
        raise RuntimeError(f"{LOCAL_ROOT}/Files not found: is lh_retail attached as default lakehouse?")

    cfg = GeneratorConfig(
        seed=int(seed), history_days=int(history_days), initial_customers=int(initial_customers),
        initial_products=int(initial_products), orders_per_day=int(orders_per_day),
        dq_error_rate=float(dq_error_rate),
    )
    run.log.info("config: %s", cfg)

    summary = run_generator(LOCAL_ROOT, LANDING_ROOT, STATE_FILE, run.run_date, cfg, log=run.log)

    run.metric("days_generated", summary["days"])
    run.metric("first_day", summary["first_day"])
    run.metric("last_day", summary["last_day"])
    run.metric("files_written", summary["files"])
    for entity, n in summary["rows"].items():
        run.metric(f"rows_{entity}", n)

## Verify: what landed?

In [ ]:
for entity in ENTITIES:
    days = sorted(f.name for f in notebookutils.fs.ls(f"{LANDING_ROOT}/{entity}"))
    print(f"{entity:12s} {len(days):3d} day folders  {days[0]} … {days[-1]}")

In [ ]:
# Peek at the latest orders file with Spark: everything is still raw text at this point
latest = sorted(f.name for f in notebookutils.fs.ls(f"{LANDING_ROOT}/orders"))[-1]
display(spark.read.option("header", True).csv(f"{LANDING_ROOT}/orders/{latest}").limit(20))

In [ ]:
notebookutils.notebook.exit(run.exit_value())